# Using Python to Access and Plot EQ Synapse Data

## Setup

In [ ]:
# Standard libraries
from datetime import datetime
from math import *
from pathlib import Path

# Installed libraries
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from matplotlib.dates import DateFormatter
%matplotlib inline

# EQ libraries
from equser.data import load_cpow_scaled, parse_start_time, SAMPLE_RATE_HZ
from equser.analysis.waveform import plot_extracted_cycles
from equser.widgets import create_file_selector

# Create output folder (adjust as needed)
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

# Base data directory (adjust as needed)
data_dir = Path('/mnt/eqdata/') # SD card

# Equipment name, for plotting below
equipment_name = "Hitachi Etcher"

# Sensor power configuration
wye = True # True for wye, False for delta

## Power Monitor Data

### Select the file

Run the code block below, then select a file from the list list and continue.

> 📝 **Note**<br>
> Power monitor files are named with the UTC year, month, day, hour, and minute in the form of `YYYYMMDD_HHmm.parquet`.

In [ ]:
pmon_dir = data_dir / 'pmon'
file_selector = create_file_selector(pmon_dir)
display(file_selector)

In [ ]:
# Get the selected file from the dialog
pmon_file = file_selector.get_selected()
# Or override the file selector to directly set the file, for example:
# pmon_file = pmon_dir / '20241111_0335.parquet'

### Load the data
Load the data and extract the time column:

In [ ]:
print(f"Loading data from {pmon_file}\n")
pmon_table = pq.read_table(pmon_file)
time_pmon = np.array(pmon_table['time_us'], dtype='datetime64[us]')
print("Available metrics:")
print(", ".join(name for name in pmon_table.column_names if name != 'time_us'))

> 📝 **Note**<br>
> In a 3-wire Delta power system configuration, the RMS voltage metrics (`AVRMS`, `BVRMS`, `CVRMS`, `AFVRMS`, `BFVRMS`, `CFVRMS`) show different values depending on the sensor configuration:
> - **If configured for Delta measurement**: These RMS metrics display line-to-line voltages (A→AB, B→AC, C→CB)
> - **If configured for Wye measurement**: These RMS metrics may be misleading because they lack a suitable voltage reference point.
>
> Also note that in a Delta power system the per-phase power metrics (`AWATT`, `BWATT`, `CWATT`, `AFWATT`, `BFWATT`, `CFWATT`, `AFVAR`, `BFVAR`, `CFVAR`) are only accurate if the CTs are installed on the respective line-to-line paths.

### Plot the data

Create a plot from the available metrics. First we will choose RMS voltage:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("RMS Voltages", x=0.53)
ax.set_title(f"File {pmon_file.stem} @ {equipment_name}", loc='center', fontsize=8)
if wye:
    ax.plot(time_pmon, np.array(pmon_table['AVRMS']), 'k', label='A', zorder=3, alpha=0.7)
    ax.plot(time_pmon, np.array(pmon_table['BVRMS']), 'r', label='B', zorder=2, alpha=0.7)
    ax.plot(time_pmon, np.array(pmon_table['CVRMS']), 'b', label='C', zorder=1, alpha=0.7)
else:
    ax.plot(time_pmon, np.array(pmon_table['AVRMS']), 'k', label='AB', zorder=3, alpha=0.7)
    ax.plot(time_pmon, np.array(pmon_table['CVRMS']), 'r', label='BC', zorder=2, alpha=0.7)
    ax.plot(time_pmon, np.array(pmon_table['BVRMS']), 'b', label='CA', zorder=1, alpha=0.7)
ax.set_ylabel("RMS Voltage (V)")
ax.set_xlabel("Time (UTC)")
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M:%S'))
ax.legend(loc='upper right')
plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.savefig(plot_dir / f"{pmon_file.stem}_voltage.svg")
plt.show()
plt.close()

Now plot active power.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("Power", x=0.53)
ax.set_title(f"File {pmon_file.stem} @ {equipment_name}", loc='center', fontsize=8)
ax.plot(time_pmon, np.array(pmon_table['AWATT']), 'k', label='A', zorder=3, alpha=0.7)
ax.plot(time_pmon, np.array(pmon_table['BWATT']), 'r', label='B', zorder=2, alpha=0.7)
ax.plot(time_pmon, np.array(pmon_table['CWATT']), 'b', label='C', zorder=1, alpha=0.7)
total_active_power = np.array(pmon_table['AWATT']) + np.array(pmon_table['BWATT']) + np.array(pmon_table['CWATT'])
ax.plot(time_pmon, total_active_power, 'k', label='Total', zorder=4)
ax.set_ylabel("Active Power (W)")
ax.set_xlabel("Time (UTC)")
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(DateFormatter('%m-%d %H:%M:%S'))
ax.legend(loc='upper right')
plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.savefig(plot_dir / f"{pmon_file.stem}_power.svg")
plt.show()
plt.close()

## Waveform Data

### Select the file

Run the code block below, then select a file from the pull-down list and continue.

> 📝 **Note**<br>
> Waveform or continuous point-on-wave (CPOW) files are named with the UTC year, month, day, hour, minute, and second in the form of `YYYYMMDD_HHmmss.parquet`.

In [ ]:
cpow_dir = data_dir / 'cpow'
file_selector = create_file_selector(cpow_dir)
display(file_selector)

In [ ]:
# Get the selected file from the dialog
cpow_file = file_selector.get_selected()
# Or override the file selector to directly set the file, for example:
#cpow_file = cpow_dir / '20241111_035139.parquet'

### Load the data

In [ ]:
print(f"Loading data from {cpow_file}\n")
cpow_table = pq.read_table(cpow_file)
print("Available signals:")
print(", ".join(cpow_table.column_names))

Load the voltage and current scaling factors:

In [ ]:
parquet_file = pq.ParquetFile(cpow_file)
user_metadata = parquet_file.metadata.metadata
vscale = float(user_metadata[b'vscale'].decode()) if b'vscale' in user_metadata else 1.0
iscale = float(user_metadata[b'iscale'].decode()) if b'iscale' in user_metadata else 1.0

Extract voltage and time:

In [ ]:
# Line-to-reference voltages
VA = cpow_table['VA'].to_numpy() * vscale
VB = cpow_table['VB'].to_numpy() * vscale
VC = cpow_table['VC'].to_numpy() * vscale

# Line-to-line voltages
VAB = VA - VB
VBC = VB - VC
VCA = VC - VA

if wye:
    signal_dict = {'VA': VA, 'VB': VB, 'VC': VC}
else:
    signal_dict = {'VAB': VA - VB, 'VBC': VB - VC, 'VCA': VC - VA}

# Get the time base
epoch_start_time = parse_start_time(user_metadata[b'start_time'].decode('utf-8'))
time_cpow = np.arange(0, len(VA)/SAMPLE_RATE_HZ, 1/SAMPLE_RATE_HZ)

### Plot the data

Plot the voltage waveforms for selected zero-crossing windows:

In [ ]:
fig, axes, window_times = plot_extracted_cycles(signal_dict, time_cpow,
                                                [0, 10, 30], # Seconds into the file to start plotting
                                                num_cycles=2, # Number of cycles to plot each window
                                                epoch_start_time=epoch_start_time)
fig.suptitle(f"Voltage Waveforms — Selected Cycles\nFile {cpow_file.stem} @ {equipment_name}", x=0.53)
plt.tight_layout()
plt.subplots_adjust(top=0.86, bottom = 0.14)
plt.savefig(plot_dir / f"{cpow_file.stem}_voltage.svg")
plt.show()
plt.close()

Plot the current waveforms:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle(f"Current Waveforms — Selected Time\nFile {cpow_file.stem} @ {equipment_name}", x=0.53)
ax.set_xlabel("Elapsed time (ms)")
ax.set_ylabel("Current (A)")
our_slice = slice(0, 1500) # start and end indices (each index is 1/32000 s)

ax.plot(time_cpow[our_slice]*1000, cpow_table['IA'][our_slice].to_numpy() * iscale, 'k-', label='IA', alpha=0.8)
ax.plot(time_cpow[our_slice]*1000, cpow_table['IB'][our_slice].to_numpy() * iscale, 'r-', label='IB', alpha=0.8)
ax.plot(time_cpow[our_slice]*1000, cpow_table['IC'][our_slice].to_numpy() * iscale, 'b-', label='IC', alpha=0.8)
ax.legend()
plt.tight_layout()
plt.subplots_adjust(top=0.9, bottom = 0.14)
plt.savefig(plot_dir / f"{cpow_file.stem}_current.svg")
plt.show()
plt.close()